# Day 2 · Exercise 4: When should the factory run the production line?

**What you'll be able to say by the end**
1. "Running the line every week costs **$24,000**. Running it in big batches costs $17,050. The MIP plan runs it 6 times for **$15,150**."
2. "Each week's stock = last week's stock + what we make − demand. This **inventory balance** is what links the weeks together."
3. "With a huge big-M, the LP relaxation claims the plan could cost **$29**, which is useless. Keep M as small as is safe."
4. "Higher setup cost means fewer runs and more stock. That's the same tradeoff as EOQ, which is Day 3."

**New ideas today:** multi-period planning, inventory balance, setup (fixed) costs over time, and choosing big-M well.

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


## 1. The story

A factory makes one running shoe. Planning gives us a **12-week demand forecast** (this is where your forecasting work plugs in).

Two costs pull in opposite directions:
- **Setup cost, $2,000** each week the line runs: changeover, cleaning, test pairs. It's the same whether we make 100 pairs or 3,000.
- **Holding cost, $0.50** per pair left in the warehouse at the end of each week: space, capital, risk of damage.

The line can make at most **3,000 pairs a week**. We start and end with an empty warehouse, and we can't run out.

**The question:** in which weeks should we run the line, and how many pairs should we make each time?

(The cost of making each pair doesn't matter here: we make 14,500 pairs in total whatever plan we pick, so it's the same for every plan.)

In [2]:
import math

import pandas as pd
import plotly.graph_objects as go
import pulp
from plotly.subplots import make_subplots

WEEKS = list(range(1, 13))
forecast = dict(zip(WEEKS, [1200, 900, 1500, 800, 600, 1800, 2200, 1000, 700, 1300, 1600, 900]))  # pairs
setup_cost = 2_000     # $ each week the line runs
holding_cost = 0.50    # $ per pair left in stock at the end of a week
line_capacity = 3_000  # pairs per week

fig = go.Figure(go.Bar(x=WEEKS, y=list(forecast.values()), marker_color="#BAB0AC",
                       hovertemplate="week %{x}<br>%{y:,} pairs<extra></extra>"))
fig.add_hline(y=line_capacity, line_dash="dot", line_color="gray", annotation_text="line capacity")
fig.update_layout(title=f"Demand forecast: {sum(forecast.values()):,} pairs over 12 weeks",
                  xaxis=dict(title="week", dtick=1), yaxis_title="pairs", height=380, width=1000, template="plotly_white")
fig.show()

## 2. Two plans by hand

**Plan A, run every week:** make exactly each week's demand. Stock is always zero, so holding costs nothing, but we pay 12 setups.

**Plan B, big batches:** whenever stock can't cover this week, run the line flat out (3,000 pairs). Few setups, but lots of stock sitting around.

To cost a plan we need to track stock week by week, using the rule:

> **end-of-week stock = start-of-week stock + pairs made − demand**

That rule is called the **inventory balance**. It's the "what comes in − what goes out" idea from Day 1's hub, applied to time instead of places: the warehouse carries stock from one week into the next.

In [3]:
def plan_table(make, demand=forecast):
    """Walk the inventory balance week by week: end stock = start stock + make - demand."""
    rows, stock = [], 0
    for t in WEEKS:
        start = stock
        stock = start + make[t] - demand[t]
        rows.append({"week": t, "start stock": start, "make": make[t], "demand": demand[t], "end stock": stock})
    return pd.DataFrame(rows).set_index("week")


def plan_cost(make, setup_cost=setup_cost):
    table = plan_table(make)
    assert (table["end stock"] >= 0).all(), "plan runs out of stock"
    runs = sum(1 for t in WEEKS if make[t] > 0)
    return {"runs": runs, "setup $": runs * setup_cost, "holding $": holding_cost * table["end stock"].sum(),
            "total $": runs * setup_cost + holding_cost * table["end stock"].sum()}


def big_batches():
    """Hand rule: whenever stock can't cover this week, run the line flat out."""
    make, stock = {}, 0
    remaining = sum(forecast.values())
    for t in WEEKS:
        if stock < forecast[t]:
            make[t] = min(line_capacity, remaining - stock)  # don't make more than is still needed
        else:
            make[t] = 0
        stock += make[t] - forecast[t]
        remaining -= forecast[t]
    return make


every_week = dict(forecast)
batches = big_batches()
pd.DataFrame({"A: run every week": plan_cost(every_week), "B: big batches": plan_cost(batches)}).T

,runs,setup $,holding $,total $
A: run every week,12.0,24000.0,0.0,24000.0
B: big batches,5.0,10000.0,7050.0,17050.0


Here's Plan B week by week, so you can see the inventory balance at work:

In [4]:
plan_table(batches)

,start stock,make,demand,end stock
week,,,,
1,0,3000,1200,1800
2,1800,0,900,900
3,900,3000,1500,2400
4,2400,0,800,1600
5,1600,0,600,1000
6,1000,3000,1800,2200
7,2200,0,2200,0
8,0,3000,1000,2000
9,2000,0,700,1300


Week 1 starts empty, makes 3,000 and sells 1,200, so it ends with 1,800. Week 2 starts with 1,800, makes nothing and sells 900, so it ends with 900. And so on.

| | Runs | Setup | Holding | Total |
|---|---|---|---|---|
| A: run every week | 12 | $24,000 | $0 | **$24,000** |
| B: big batches | 5 | $10,000 | $7,050 | **$17,050** |

Plan B is much better, but it carries a lot of stock. Week 3, for example, ends with 2,400 pairs. Is there a plan in between that beats both? That's a MIP.

## 3. Write it as a MIP

| Piece | Plain words | Here |
|---|---|---|
| **Binary variables** | Yes/no each week | `run[t]` = 1 if the line runs in week *t* |
| **Continuous variables** | Quantities each week | `make[t]` = pairs made; `stock[t]` = pairs left at the end of week *t* |
| **Objective** | Minimize | Σ ($2,000 × `run[t]` + $0.50 × `stock[t]`) |
| **Inventory balance** | Links each week to the next | `stock[t−1]` + `make[t]` − demand[t] = `stock[t]` |
| **Big-M (run link)** | Only make pairs in weeks the line runs | `make[t]` ≤ M × `run[t]` |
| **Capacity** | The line's limit | `make[t]` ≤ 3,000 |
| **No stockouts** | Demand must be met from stock | `stock[t]` ≥ 0 (stock can't go negative) |

Notice that **"no stockouts" needs no extra rule**. Because stock can't be negative, the balance rule forces us to make enough, early enough.

The run link is the same big-M trick as the DC linking rule in exercise 3. We'll look at how to choose M in step 5; for now, M = line capacity.

In [5]:
def solve(setup_cost=setup_cost, big_m="capacity", relax=False):
    """Lot sizing MIP. big_m: 'huge' (1,000,000), 'capacity', or 'tight' (min of capacity, demand still to come)."""
    model = pulp.LpProblem("lot_sizing", pulp.LpMinimize)

    # Decision variables
    run = {t: model.add_variable(f"run_{t}", lowBound=0, upBound=1,
                                 cat="Continuous" if relax else "Binary") for t in WEEKS}  # 1 = run the line
    make = {t: model.add_variable(f"make_{t}", lowBound=0) for t in WEEKS}                # pairs made
    stock = {t: model.add_variable(f"stock_{t}", lowBound=0) for t in WEEKS}              # pairs left at week end

    # Objective: setups + holding
    model += pulp.lpSum(setup_cost * run[t] + holding_cost * stock[t] for t in WEEKS)

    for t in WEEKS:
        # Inventory balance: last week's stock + what we make - demand = this week's stock
        previous = stock[t - 1] if t > 1 else 0
        model += previous + make[t] - forecast[t] == stock[t], f"balance_{t}"

        # Big-M: can only make pairs in a week the line runs
        M = {"huge": 1_000_000,
             "capacity": line_capacity,
             "tight": min(line_capacity, sum(forecast[u] for u in WEEKS if u >= t))}[big_m]
        model += make[t] <= M * run[t], f"run_link_{t}"

        # Line capacity
        model += make[t] <= line_capacity, f"capacity_{t}"

    stats = model.solve(pulp.HiGHS(msg=False))
    assert stats.status == pulp.LpSolveStatus.Optimal, stats.status
    return model, run, make, stock


model, run, make, stock = solve()
best_make = {t: round(make[t].varValue) for t in WEEKS}
print(f"Best plan: ${pulp.value(model.objective):,.0f}")
pd.Series(plan_cost(best_make))

Best plan: $15,150


runs             6.0
setup $      12000.0
holding $     3150.0
total $      15150.0
dtype: float64

In [6]:
def plan_chart(make, title):
    table = plan_table(make)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=WEEKS, y=table["demand"], name="demand (forecast)", marker_color="#BAB0AC",
                         hovertemplate="week %{x}<br>demand %{y:,}<extra></extra>"))
    fig.add_trace(go.Bar(x=WEEKS, y=table["make"], name="make", marker_color="#4C78A8",
                         hovertemplate="week %{x}<br>make %{y:,}<extra></extra>"))
    fig.add_trace(go.Scatter(x=WEEKS, y=table["end stock"], name="end-of-week stock", mode="lines+markers",
                             line=dict(color="#F58518", width=3),
                             hovertemplate="week %{x}<br>stock %{y:,}<extra></extra>"))
    fig.add_hline(y=line_capacity, line_dash="dot", line_color="gray", annotation_text="line capacity")
    fig.update_layout(title=title, barmode="group", xaxis=dict(title="week", dtick=1), yaxis_title="pairs",
                      height=430, width=1000, template="plotly_white", legend=dict(orientation="h", y=1.1))
    return fig


plan_chart(best_make, f"Best plan: {plan_cost(best_make)['runs']} runs, ${pulp.value(model.objective):,.0f}").show()
plan_table(best_make)

,start stock,make,demand,end stock
week,,,,
1,0,2100,1200,900
2,900,0,900,0
3,0,2900,1500,1400
4,1400,0,800,600
5,600,0,600,0
6,0,2000,1800,200
7,200,3000,2200,1000
8,1000,0,1000,0
9,0,2000,700,1300


**Reading the plan.** Each run covers a little block of weeks:

| Run in week | Makes | Covers |
|---|---|---|
| 1 | 2,100 | weeks 1–2 |
| 3 | 2,900 | weeks 3–5 |
| 6 | 2,000 | week 6, plus 200 pairs for week 7 |
| 7 | 3,000 (full) | rest of week 7, and week 8 |
| 9 | 2,000 | weeks 9–10 |
| 11 | 2,500 | weeks 11–12 |

**Why join weeks into one run?** Week 3's run also makes weeks 4 and 5. That means holding 800 pairs for 1 week ($400) and 600 pairs for 2 weeks ($600): $1,000 of holding to save a $2,000 setup. Worth it.

**Why not join week 6 into week 3's run?** That run would need 2,900 + 1,800 = 4,700 pairs, over the 3,000 capacity. Even without that limit, 1,800 pairs sitting for 3 weeks is $2,700 of holding to save a $2,000 setup. Not worth it.

**Why run in both week 6 and week 7?** Weeks 7 and 8 together need 3,200 pairs, but the line can only make 3,000 in a week. So 200 pairs have to be made a week early. Capacity forces this, and the MIP handles it automatically.

**Result: $15,150**, which beats both hand plans ($24,000 and $17,050). The MIP isn't following a single rule like "batch three weeks". It weighs every setup against every pair-week of holding at once.

## 4. Choosing big-M: why "big enough" isn't good enough

The run link `make[t] ≤ M × run[t]` works for **any** M that's at least as big as the most we could ever make in a week. A lazy modeler writes M = 1,000,000 "to be safe". Every choice gives the same real answer. But they give very different **LP relaxations** (exercise 3: the relaxation is what branch and bound uses to prune).

Three choices:
- **huge:** M = 1,000,000
- **capacity:** M = 3,000, the line's real limit
- **tight:** M = the smaller of capacity and *all the demand still to come*. In week 12 there's only 900 pairs of demand left, so there's never a reason to make more than 900.

In [7]:
mip = pulp.value(model.objective)
rows = []
for big_m in ["huge", "capacity", "tight"]:
    relaxed, *_ = solve(big_m=big_m, relax=True)  # keep the model alive: PuLP 4 frees it otherwise
    exact, *_ = solve(big_m=big_m)
    lp = pulp.value(relaxed.objective)
    rows.append({"M": big_m, "LP relaxation (lower bound) $": round(lp),
                 "real best $": round(pulp.value(exact.objective)), "gap %": round(100 * (mip - lp) / mip, 1)})
pd.DataFrame(rows).set_index("M")

,LP relaxation (lower bound) $,real best $,gap %
M,,,
huge,29,15150,99.8
capacity,9667,15150,36.2
tight,10450,15150,31.0


**All three find the same real plan ($15,150).** But look at the lower bounds:

- **Huge M: $29.** In the relaxation, `run` can be a fraction. To make 1,200 pairs, it only needs `run` = 1,200 ÷ 1,000,000 = 0.0012, which "pays" 0.0012 × $2,000 = $2.40 of setup. Across all 14,500 pairs, the relaxation pays just $29 in setups. A lower bound of $29 on a $15,150 problem tells branch and bound nothing, so it can't prune anything.
- **Capacity: $9,667.** Now `run` must be at least make ÷ 3,000, so the relaxation pays real money for setups.
- **Tight: $10,450.** Late weeks have small M, so their `run` values are pushed close to 1.

**Why a huge M hurts, in interview terms:**
1. **Weak bounds.** The relaxation is far from reality, so branch and bound prunes almost nothing and the solver gets slow on bigger problems.
2. **Numerical trouble.** Solvers treat numbers within about 0.000001 of zero as zero. With M = 1,000,000, a "zero" `run` of 0.000001 still lets 1 pair through. With M = 10⁹, it lets 1,000 pairs through, so the "closed" line quietly produces.

**Rule:** make M the smallest number that never cuts off a real plan.

## 5. Change one input: the setup cost

Step 4 of the daily loop: change something, predict what happens, then check. **Prediction:** as setups get more expensive, the plan should run the line less often and carry more stock.

In [8]:
rows = []
for k in [0, 250, 500, 1000, 1500, 2000, 3000, 4000, 6000, 8000]:
    m_k, _, make_k, stock_k = solve(setup_cost=k)
    mk = {t: round(make_k[t].varValue) for t in WEEKS}
    rows.append({"setup $": k, "runs": sum(1 for t in WEEKS if mk[t] > 0),
                 "avg stock (pairs)": round(sum(stock_k[t].varValue for t in WEEKS) / len(WEEKS)),
                 "total $": round(pulp.value(m_k.objective))})
sweep = pd.DataFrame(rows)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=sweep["setup $"], y=sweep["runs"], name="line runs", mode="lines+markers",
                         line_shape="hv", hovertemplate="setup $%{x:,}<br>%{y} runs<extra></extra>"), secondary_y=False)
fig.add_trace(go.Scatter(x=sweep["setup $"], y=sweep["avg stock (pairs)"], name="average stock", mode="lines+markers",
                         hovertemplate="setup $%{x:,}<br>avg stock %{y:,} pairs<extra></extra>"), secondary_y=True)
fig.add_vline(x=setup_cost, line_dash="dot", line_color="gray", annotation_text="today: $2,000")
fig.update_xaxes(title_text="setup cost ($ per run)", tickprefix="$")
fig.update_yaxes(title_text="runs in 12 weeks", secondary_y=False)
fig.update_yaxes(title_text="average stock (pairs)", secondary_y=True)
fig.update_layout(title="Higher setup cost: fewer runs, more stock", height=430, width=1000,
                  template="plotly_white", legend=dict(orientation="h", y=1.1))
fig.show()
sweep

,setup $,runs,avg stock (pairs),total $
0,0,12,0,0
1,250,12,0,3000
2,500,8,258,5550
3,1000,6,525,9150
4,1500,6,525,12150
5,2000,6,525,15150
6,3000,6,525,21150
7,4000,5,1175,27050
8,6000,5,1175,37050
9,8000,5,1175,47050


**The prediction holds:**
- **Setups at $0–250:** run every week and hold nothing. Setups are cheaper than carrying stock.
- **$1,000–3,000:** the plan settles at 6 runs. Small changes in setup cost don't change the plan, only its total cost. This is the same "stable over a range" pattern as the shadow prices in Day 1.
- **$4,000 and up:** 5 runs, and it can't go lower. 14,500 pairs ÷ 3,000 per run = 4.83, so we need **at least 5 runs** whatever setups cost. Capacity sets the floor.

**Preview of Day 3:** the classic **EOQ** formula gives the best batch size when demand is steady:

> EOQ = √(2 × demand per week × setup cost ÷ holding cost per week) = √(2 × 1,208 × 2,000 ÷ 0.50) ≈ **3,100 pairs**

That's about 2.6 weeks of demand per batch, or 4 to 5 runs in 12 weeks. The MIP runs 6, because real demand is lumpy (week 7 is 2,200, week 5 is 600) and capacity caps each run at 3,000. EOQ is the rule of thumb; the MIP handles the messy real version.

**Where your forecast matters:** this whole plan assumes the forecast is right. If week 7 comes in at 3,000 instead of 2,200, we run out. Days 3 and 4 add **safety stock** and plan against **many demand scenarios** at once.

## Recap: say these out loud

1. **What's lot sizing?** Deciding when to produce and how much, trading setup cost (produce often) against holding cost (produce in big batches).
2. **What links the weeks?** Inventory balance: stock at the end of a week = stock at the start + pairs made − demand. No stockouts = stock ≥ 0.
3. **How is a setup modeled?** A binary `run[t]`, $2,000 × `run[t]` in the objective, and `make[t]` ≤ M × `run[t]`.
4. **What goes wrong if M is too large?** The LP relaxation gets weak ($29 vs a real $15,150), so the solver prunes less and slows down; and you can get numerical errors where a "closed" line still produces.
5. **How do you pick M?** The smallest value that never cuts off a real plan: here, the smaller of capacity and remaining demand.
6. **What did the setup-cost sweep show?** Higher setup cost means fewer runs and more stock. The plan is stable over ranges, and capacity sets a floor of 5 runs.

**Words from today**

| Word | Meaning |
|---|---|
| Lot sizing | Choosing production timing and quantities over many periods |
| Setup cost | Fixed cost each time production starts, whatever the quantity |
| Holding cost | Cost per unit per period of carrying stock |
| Inventory balance | start stock + in − out = end stock, for every period |
| Big-M | The M in "quantity ≤ M × binary". Should be as small as is safe |
| Planning horizon | How many periods the plan looks ahead (12 weeks here) |
| EOQ | Economic order quantity: the best batch size when demand is steady |

**Try it yourself:**
1. Change `holding_cost` to 1.00. *Predict:* more runs or fewer? Check with `solve()`.
2. Change week 7's forecast to 3,000. *Predict:* which week has to pre-build pairs now, and how many?